# W10 — retrain the adapters on the normalised bundle, with and without RVC

Closes the results before the freeze. Four LoRA adapters, each identical to the
original except for the data it reads:

| Run | Data | Condition |
|---|---|---|
| `lora_norm_clean` | XTTS | clean |
| `lora_norm_channel` | XTTS | G.711 @ 20 dB |
| `lora_norm_rvc_clean` | XTTS + RVC training half | clean |
| `lora_norm_rvc_channel` | XTTS + RVC training half | G.711 @ 20 dB |

**Why retrain.** The first bundles were never level-normalised, so the shortcut
gate separated real from fake at 1.39% EER on level alone and the clean 1.34% was
not quotable (`lowlevel_cue_check_v1.md`). The bundle is now normalised to
−23 dBFS (`src/data/normalise_bundle.py`). The gate still fails afterwards, at
5.17% clean and 10.01% channel-matched (`bundle_normalisation_v1.md`), so read
every EER below against those floors, not against 50%. The channel-matched
adapters carry the headline.

**Why RVC.** Scoring showed the XTTS-trained adapter waves 76–83% of RVC fakes
through as real (P-025). The RVC half used here is speaker-held-out: every test
conversion's target *and* source speaker is absent from training
(`src/data/rvc_holdout.py`), so RVC detection is still measurable.

Each adapter is then scored, in its own condition, on the eval pool, on CM04
(Tortoise, never seen) and on the RVC test half.

---

## Before you run

1. **Accelerator → GPU T4 x2. Internet → On.**
2. **+ Add Input → `saikrishnareddy9/codemix-bundle-normalised`** — holds a
   `clean/` root and a `channel/` root.
3. No ethics PDF is needed: no audio is synthesised here.

About 5–10 min per adapter on a T4, plus a few minutes of scoring each. Run with
`SMOKE = True` first.

## 1. Configuration

In [ ]:
REPO_HOST = "github.com/Mounika-Reddy-0802/codemix-deepfake-detection.git"
BRANCH = "week10-mounika-unseen-attack-scoring"   # switch to "main" once merged

SMOKE = True          # True: 1% subset, confirms the chain; then set False and Save & Run All

RUNS = [
    # (name, config, condition)
    ("lora_norm_clean",       "configs/train_lora_norm_clean.yaml",       "clean"),
    ("lora_norm_channel",     "configs/train_lora_norm_channel.yaml",     "channel"),
    ("lora_norm_rvc_clean",   "configs/train_lora_norm_rvc_clean.yaml",   "clean"),
    ("lora_norm_rvc_channel", "configs/train_lora_norm_rvc_channel.yaml", "channel"),
]

EVALS = {
    "clean":   {"eval_pool": "data/manifests/codemix_eval.csv",
                "cm04":      "data/manifests/score_cm04_norm.csv",
                "rvc_test":  "data/manifests/rvc_holdout_test.csv"},
    "channel": {"eval_pool": "data/manifests/codemix_eval_channel20.csv",
                "cm04":      "data/manifests/score_cm04_norm_channel20.csv",
                "rvc_test":  "data/manifests/rvc_holdout_test_channel20.csv"},
}

## 2. Clone the repo and pull the Stage-1 checkpoint (every adapter starts from it)

In [ ]:
import glob, json, os, shutil, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working")
REPO = WORK / "codemix-deepfake-detection"


def sh(cmd, check=True):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, cwd=REPO if REPO.exists() else None)
    if r.stdout.strip():
        print(r.stdout[-2500:])
    if r.returncode and r.stderr.strip():
        print(r.stderr[-2500:])
    if check and r.returncode:
        raise SystemExit(f"failed ({r.returncode}): {cmd}")
    return r


if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", f"https://{REPO_HOST}", str(REPO)], check=True)
os.chdir(REPO)
sh("apt-get -qq install -y git-lfs", check=False)
sh("git lfs install --local && git lfs pull --include checkpoints/baseline/best.pt")
size = (REPO / "checkpoints/baseline/best.pt").stat().st_size
assert size > 10_000_000, f"baseline checkpoint is an LFS pointer ({size} bytes) -- re-run this cell"
print("baseline checkpoint OK:", round(size / 1e6), "MB")

## 3. Locate the two normalised roots

In [ ]:
clean = glob.glob("/kaggle/input/**/clean/clips", recursive=True)
channel = glob.glob("/kaggle/input/**/channel/clips", recursive=True)
if not clean or not channel:
    for p in sorted(glob.glob("/kaggle/input/*/*")):
        print("  ", p)
    raise SystemExit("attach saikrishnareddy9/codemix-bundle-normalised (needs clean/ and channel/)")
ROOT = {"clean": str(Path(clean[0]).parent), "channel": str(Path(channel[0]).parent)}
print(ROOT)

import pandas as pd
from src.data.scoring_manifests import missing_files

for cond, manifests in EVALS.items():
    for name, m in manifests.items():
        gone = missing_files(pd.read_csv(m), ROOT[cond])
        print(f"{cond:8s} {name:10s} missing {len(gone)}")
        assert not gone, f"{m}: {gone[:3]}"

## 4. Anti-leakage gate — runs before any training, per the team rules

In [ ]:
sh(f"{sys.executable} -m pip install -q 'transformers>=4.57,<5' soundfile==0.12.1 librosa==0.11.0", check=False)
sh(f"{sys.executable} -m pytest tests/test_splits.py tests/test_rvc_holdout.py tests/test_lora.py -q")

## 5. Train the four adapters

In [ ]:
import time

for name, cfg, cond in RUNS:
    t0 = time.time()
    sh(f"{sys.executable} -m src.training.train --config {cfg} --data-root {ROOT[cond]} --device cuda"
       + (" --smoke" if SMOKE else ""))
    print(f"[{name}] trained in {(time.time() - t0) / 60:.1f} min")

## 6. Score each adapter in its own condition

In [ ]:
OUT = WORK / "w10_results"
OUT.mkdir(exist_ok=True)
results = {}
for name, cfg, cond in RUNS:
    ckpt = REPO / "checkpoints" / name / "best.pt"
    for ev, manifest in EVALS[cond].items():
        tag = f"{name}__{ev}"
        sh(f"{sys.executable} -m src.training.evaluate --checkpoint {ckpt} --manifest {manifest} "
           f"--data-root {ROOT[cond]} --device cuda --batch-size 32 --num-workers 4 --max-seconds 4.0 "
           f"--scores-out {OUT}/{tag}_scores.csv --out {OUT}/{tag}.json"
           + (" --limit 200" if SMOKE else ""))
        p = json.loads((OUT / f"{tag}.json").read_text()).get("pooled", {})
        results[tag] = p
        print(f"{tag:40s} EER {p.get('eer', float('nan')) * 100:6.2f}%")

## 7. The table, and what to download

In [ ]:
rows = [{"adapter": k.split("__")[0], "set": k.split("__")[1],
         "EER %": round(v["eer"] * 100, 2), "AUC": round(v["auc"], 4), "clips": v.get("clips")}
        for k, v in results.items()]
table = pd.DataFrame(rows).pivot(index="adapter", columns="set", values="EER %")
print(table.to_string())
(OUT / "w10_summary.json").write_text(json.dumps(results, indent=2))
for name, _, _ in RUNS:
    shutil.copy(REPO / "checkpoints" / name / "best.pt", OUT / f"{name}_best.pt")
print("\nDownload w10_results/ from the Output tab: JSONs + per-clip scores + the four best.pt files.")